# Classification Pipeline Exercise: Reproducible Growth-Status Modelling

This notebook adapts the classification exercise workflow to the Cane Corso Growth Intelligence project.

The goal is not only to train another classifier. The goal is to practice a complete and reproducible classification workflow:

1. load and inspect the data
2. create dummy baselines
3. build preprocessing pipelines
4. use cross-validation
5. analyze learning curves
6. engineer useful features
7. compare models
8. use safer feature importance analysis
9. perform error analysis
10. run an ablation-style comparison

The task remains educational. The model does **not** provide veterinary diagnosis.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    make_scorer,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42


## Problem 1 — Load and Inspect the Data

For this exercise, I use the balanced classification sample:

`data/processed/dog_growth_classification_sample.csv`

The target is already prepared:

- `growth_status`: text label
- `growth_status_binary`: numeric binary target

The target was derived from body-condition categories. Therefore, body-condition columns are **not used as model input features**, because that would leak the answer into the model.

For classroom speed and reproducibility, this notebook uses a balanced modelling subset from the processed sample. The full processed classification sample remains available in the project.


In [ ]:
data_path = "../data/processed/dog_growth_classification_sample.csv"
raw_df = pd.read_csv(data_path)

print("Full classification sample shape:", raw_df.shape)
print(raw_df["growth_status"].value_counts())
raw_df.head()


In [ ]:
# Balanced modelling subset for faster exercise runs.
# This keeps the notebook practical while preserving both classes.
rows_per_class_for_exercise = 2500

df = (
    raw_df
    .groupby("growth_status_binary", group_keys=False)
    .sample(n=rows_per_class_for_exercise, random_state=RANDOM_STATE)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print("Modelling subset shape:", df.shape)
print(df["growth_status"].value_counts())
df.head()


In [ ]:
df.info()


## Experimental Protocol

To keep the exercise reproducible, I define the modelling protocol before comparing models.

The protocol is:

- use the same target: `growth_status_binary`
- exclude leakage columns such as `bcs_recorded`, `bcs_predicted`, `bcs_source`, `growth_status`, and `growth_status_binary`
- use stratified 5-fold cross-validation
- compare models using the same metrics
- keep a separate holdout test split for error analysis

The main metric is F1-score because it balances precision and recall.
Recall is also important because the `needs_attention` class should not be missed in the educational setting.


In [ ]:
TARGET_COLUMN = "growth_status_binary"

base_feature_columns = [
    "visit_age_months",
    "weight_kg",
    "average_adult_breed_weight_kg",
    "gender",
    "preventive_care_visit",
    "healthy_pet_diagnosis",
]

y = df[TARGET_COLUMN].astype(int)
X_base = df[base_feature_columns].copy()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
}

X_base.head()


## Problem 2 — Dummy Baseline Models

A dummy model is a baseline that does not learn meaningful patterns.

I test two dummy strategies:

- `most_frequent`: always predicts the most common class
- `stratified`: predicts according to the observed class distribution

A real model should perform better than these baselines.


In [ ]:
def summarize_cv_results(name, cv_result):
    return {
        "Model": name,
        "Accuracy": cv_result["test_accuracy"].mean(),
        "Precision": cv_result["test_precision"].mean(),
        "Recall": cv_result["test_recall"].mean(),
        "F1-score": cv_result["test_f1"].mean(),
        "AUC": cv_result["test_roc_auc"].mean(),
    }

models_so_far = []

for strategy in ["most_frequent", "stratified"]:
    dummy = DummyClassifier(strategy=strategy, random_state=RANDOM_STATE)
    result = cross_validate(dummy, X_base, y, cv=cv, scoring=scoring, n_jobs=1)
    models_so_far.append(summarize_cv_results(f"Dummy ({strategy})", result))

pd.DataFrame(models_so_far)


## Problem 3 — Data Pipeline

The main pipeline separates numeric and categorical features.

Numeric features:

- missing values are filled with the median
- values are standardized with Z-score scaling

Categorical features:

- missing values are filled with the most frequent category
- categories are one-hot encoded

The first real model is Logistic Regression.


In [ ]:
def make_preprocessor(feature_columns):
    numeric_features = [
        column for column in feature_columns
        if column in [
            "visit_age_months",
            "weight_kg",
            "average_adult_breed_weight_kg",
            "weight_to_adult_breed_weight_ratio",
            "age_weight_ratio",
            "growth_pressure_index",
        ]
    ]

    categorical_features = [column for column in feature_columns if column not in numeric_features]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ])


def make_logistic_pipeline(feature_columns):
    return Pipeline([
        ("preprocessor", make_preprocessor(feature_columns)),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])

logistic_pipeline = make_logistic_pipeline(base_feature_columns)
logistic_cv = cross_validate(logistic_pipeline, X_base, y, cv=cv, scoring=scoring, n_jobs=1)

models_so_far.append(summarize_cv_results("Logistic Pipeline", logistic_cv))
pd.DataFrame(models_so_far)


## Problem 4 — Learning Curve

A learning curve shows how model performance changes as the training set becomes larger.

It helps answer questions such as:

- does the model benefit from more data?
- does the model have high bias?
- does the model have high variance?
- should the next improvement be more data, better features, or a different model?


In [ ]:
train_sizes, train_scores, validation_scores = learning_curve(
    logistic_pipeline,
    X_base,
    y,
    cv=cv,
    scoring="f1",
    train_sizes=np.linspace(0.2, 1.0, 5),
    shuffle=True,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

learning_curve_df = pd.DataFrame({
    "train_size": train_sizes,
    "train_f1_mean": train_scores.mean(axis=1),
    "validation_f1_mean": validation_scores.mean(axis=1),
})

learning_curve_df


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(learning_curve_df["train_size"], learning_curve_df["train_f1_mean"], marker="o", label="Training F1")
plt.plot(learning_curve_df["train_size"], learning_curve_df["validation_f1_mean"], marker="o", label="Validation F1")
plt.xlabel("Training examples")
plt.ylabel("F1-score")
plt.title("Learning Curve - Logistic Regression Pipeline")
plt.legend()
plt.show()


## Problem 5 and 6 — Feature Engineering

The original exercise extracts useful information from columns such as `Cabin` and passenger groups.

In this project, the equivalent is to create growth-related features from age, weight, and expected adult breed weight.

Engineered features:

- `weight_to_adult_breed_weight_ratio`
- `age_weight_ratio`
- `growth_pressure_index`
- `puppy_stage`
- `adult_breed_weight_group`

These features are still educational and should not be interpreted as veterinary diagnosis.


In [ ]:
def add_growth_features(dataframe):
    data = dataframe.copy()

    safe_adult_weight = data["average_adult_breed_weight_kg"].replace(0, np.nan)
    safe_age = data["visit_age_months"].replace(0, np.nan)

    data["weight_to_adult_breed_weight_ratio"] = data["weight_kg"] / safe_adult_weight
    data["age_weight_ratio"] = data["weight_kg"] / safe_age
    data["growth_pressure_index"] = data["weight_kg"] / (data["visit_age_months"] + 1)

    data["puppy_stage"] = pd.cut(
        data["visit_age_months"],
        bins=[-0.01, 6, 12, 24, 36],
        labels=["early_puppy", "late_puppy", "young_adult", "adult_transition"],
    ).astype("object")

    data["adult_breed_weight_group"] = pd.cut(
        data["average_adult_breed_weight_kg"],
        bins=[0, 10, 25, 45, 70, 200],
        labels=["small", "medium", "large", "giant", "extra_large"],
    ).astype("object")

    return data

exercise_df = add_growth_features(df)

engineered_feature_columns = base_feature_columns + [
    "weight_to_adult_breed_weight_ratio",
    "age_weight_ratio",
    "growth_pressure_index",
    "puppy_stage",
    "adult_breed_weight_group",
]

X_engineered = exercise_df[engineered_feature_columns].copy()
X_engineered.head()


In [ ]:
engineered_logistic_pipeline = make_logistic_pipeline(engineered_feature_columns)
engineered_logistic_cv = cross_validate(
    engineered_logistic_pipeline,
    X_engineered,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
)

models_so_far.append(summarize_cv_results("Logistic + Engineered Features", engineered_logistic_cv))
pd.DataFrame(models_so_far)


## Problem 7 — A Different Model

The exercise asks for another classifier with the same evaluation protocol.

I test two tree-based models:

- Random Forest Classifier
- Gradient Boosting Classifier

They are compared against the Logistic Regression pipeline using the same cross-validation strategy.


In [ ]:
def make_random_forest_pipeline(feature_columns):
    return Pipeline([
        ("preprocessor", make_preprocessor(feature_columns)),
        ("classifier", RandomForestClassifier(
            n_estimators=30,
            max_depth=8,
            random_state=RANDOM_STATE,
            class_weight="balanced",
        )),
    ])


def make_gradient_boosting_pipeline(feature_columns):
    return Pipeline([
        ("preprocessor", make_preprocessor(feature_columns)),
        ("classifier", GradientBoostingClassifier(
            n_estimators=60,
            max_depth=2,
            random_state=RANDOM_STATE,
        )),
    ])

forest_pipeline = make_random_forest_pipeline(engineered_feature_columns)
boosting_pipeline = make_gradient_boosting_pipeline(engineered_feature_columns)

forest_cv = cross_validate(forest_pipeline, X_engineered, y, cv=cv, scoring=scoring, n_jobs=1)
boosting_cv = cross_validate(boosting_pipeline, X_engineered, y, cv=cv, scoring=scoring, n_jobs=1)

models_so_far.append(summarize_cv_results("Random Forest + Engineered Features", forest_cv))
models_so_far.append(summarize_cv_results("Gradient Boosting + Engineered Features", boosting_cv))

model_comparison_df = pd.DataFrame(models_so_far)
model_comparison_df.sort_values(by="F1-score", ascending=False)


## Problem 8 — Feature Importance

Tree-based `feature_importances_` can be misleading, especially with correlated or high-cardinality features.

For this exercise, I use **permutation importance** on a holdout test split. This answers a more practical question:

> How much does model performance drop when one input feature is shuffled?

This is still model-specific and data-specific, so it should be interpreted carefully.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_engineered,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

candidate_models = {
    "Logistic + Engineered Features": engineered_logistic_pipeline,
    "Random Forest + Engineered Features": forest_pipeline,
    "Gradient Boosting + Engineered Features": boosting_pipeline,
}

holdout_results = []
for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    holdout_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
        "AUC": roc_auc_score(y_test, probabilities),
    })

holdout_results_df = pd.DataFrame(holdout_results).sort_values(by="F1-score", ascending=False)
holdout_results_df


In [ ]:
best_model_name = holdout_results_df.iloc[0]["Model"]
best_model = candidate_models[best_model_name]

importance_subset = X_test.sample(n=min(800, len(X_test)), random_state=RANDOM_STATE)
importance_y = y_test.loc[importance_subset.index]

importance_result = permutation_importance(
    best_model,
    importance_subset,
    importance_y,
    scoring="f1",
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

permutation_importance_df = pd.DataFrame({
    "Feature": importance_subset.columns,
    "Importance mean": importance_result.importances_mean,
    "Importance std": importance_result.importances_std,
}).sort_values(by="Importance mean", ascending=False)

best_model_name, permutation_importance_df


In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(
    permutation_importance_df["Feature"],
    permutation_importance_df["Importance mean"],
)
plt.xlabel("Permutation importance, F1 decrease")
plt.ylabel("Feature")
plt.title(f"Permutation Importance - {best_model_name}")
plt.gca().invert_yaxis()
plt.show()


## Problems 9 and 10 — Error Analysis

After selecting a strong candidate model, I inspect its mistakes.

The goal is not only to report a score, but to understand model behavior:

- where does the model make mistakes?
- are false positives and false negatives different?
- are some age groups harder?
- are there confidently wrong predictions?


In [ ]:
best_predictions = best_model.predict(X_test)
best_probabilities = best_model.predict_proba(X_test)[:, 1]

error_analysis_df = X_test.copy()
error_analysis_df["actual"] = y_test.values
error_analysis_df["predicted"] = best_predictions
error_analysis_df["needs_attention_probability"] = best_probabilities
error_analysis_df["is_error"] = error_analysis_df["actual"] != error_analysis_df["predicted"]

error_analysis_df["error_type"] = np.select(
    [
        (error_analysis_df["actual"] == 0) & (error_analysis_df["predicted"] == 1),
        (error_analysis_df["actual"] == 1) & (error_analysis_df["predicted"] == 0),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

error_analysis_df["error_type"].value_counts()


In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, best_predictions),
    display_labels=["normal_growth", "needs_attention"],
).plot()
plt.title(f"Holdout Confusion Matrix - {best_model_name}")
plt.show()

print(classification_report(
    y_test,
    best_predictions,
    target_names=["normal_growth", "needs_attention"],
    zero_division=0,
))


In [ ]:
error_by_stage = (
    error_analysis_df
    .groupby("puppy_stage", dropna=False)["is_error"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("error_rate")
)

error_by_stage


In [ ]:
confident_wrong = error_analysis_df[
    (error_analysis_df["is_error"])
    & (
        (error_analysis_df["needs_attention_probability"] >= 0.8)
        | (error_analysis_df["needs_attention_probability"] <= 0.2)
    )
].sort_values(by="needs_attention_probability", ascending=False)

confident_wrong.head(10)


## Problem 11 — Ablation Study

An ablation study compares feature groups under the same protocol.

This helps answer which groups add useful information.

In this project, I compare:

- numeric growth features only
- numeric + categorical base features
- ratio features only
- full engineered feature set

For each feature group, I test Logistic Regression and Random Forest.


In [ ]:
feature_group_map = {
    "numeric_base": [
        "visit_age_months",
        "weight_kg",
        "average_adult_breed_weight_kg",
    ],
    "numeric_plus_categorical_base": base_feature_columns,
    "ratio_features_only": [
        "weight_to_adult_breed_weight_ratio",
        "age_weight_ratio",
        "growth_pressure_index",
    ],
    "full_engineered": engineered_feature_columns,
}

ablation_results = []

for group_name, feature_columns in feature_group_map.items():
    X_group = exercise_df[feature_columns].copy()

    model_factories = {
        "Logistic Regression": make_logistic_pipeline,
        "Random Forest": make_random_forest_pipeline,
    }

    for model_name, factory in model_factories.items():
        model = factory(feature_columns)
        result = cross_validate(model, X_group, y, cv=cv, scoring=scoring, n_jobs=1)
        summary = summarize_cv_results(f"{model_name} | {group_name}", result)
        summary["Feature group"] = group_name
        summary["Base model"] = model_name
        ablation_results.append(summary)

ablation_df = pd.DataFrame(ablation_results)
ablation_df.sort_values(by="F1-score", ascending=False)


## Final Exercise Report

This notebook extends the first classification notebook with a more complete machine-learning workflow.

Main conclusions:

- Dummy baselines are necessary because they show the minimum performance a real model should beat.
- Pipelines make preprocessing reproducible and reduce the risk of accidental data leakage.
- Cross-validation gives a more stable estimate than a single train/test split.
- Feature engineering can improve the representation of growth-related patterns, but it must be justified.
- Permutation importance is safer than blindly trusting tree impurity-based importances.
- Error analysis is essential because a single score does not explain where the model fails.
- Ablation helps compare feature groups and shows which information is useful.

The work remains educational and does not provide veterinary diagnosis.
